<a href="https://colab.research.google.com/github/freeforall1932-design/dionysus/blob/arena/5bf33a0c-dionysus/workbench/projects/kobold-cli/kobold-cli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Kobold CLI — Dionysus coding/API edition

This notebook is adapted from the **official KoboldCpp Colab notebook**. The working upstream download, model-selection, Google Drive, tunnel-fallback, and launch mechanics are retained; the presets and defaults are changed for coding/API use.

**Fastest first try:** leave the quick preset on **Fast smoke test - Qwen2.5-Coder 1.5B**, leave the manual loader checkboxes unchecked, and press **Play** on the big cell below. The first start still has to download KoboldCpp (about 0.65 GB) and the selected model (about 1 GB for the smoke test), so it can take several minutes before the Cloudflare URL appears.

For API clients, use the printed tunnel URL plus `/v1` as the OpenAI-compatible base URL, and use the password printed by the cell as the Bearer token/API key. Do not share that password or save a copy of the notebook with cell outputs.

To choose manually instead of using a quick preset, check **LoadTextModel** (and any other upstream loader boxes you really want), then select or paste a GGUF URL.

`DeleteExistingModels` now defaults **OFF**, so a model already downloaded in `/content` during this runtime is reused instead of being deleted and downloaded again.

The upstream silent-audio keep-alive cell is intentionally omitted; it is not part of model loading. When finished, stop the big cell and use **Runtime → Disconnect and delete runtime**.

Colab terms/session limits are still the project's open W-021/Q14 check; this owner-directed test notebook is not proof that a usage pattern is permitted.


In [ ]:
# @title <b>v-- Choose coding/API settings, then click Play to start KoboldCpp</b> {"display-mode":"form"}
#@markdown You will need to restart KoboldCpp with the play button when you make changes to these settings.
#@markdown Dionysus edition: upstream download/load/launch mechanics are retained; quick presets and defaults below are coding/API-oriented.
# Adapted from LostRuins/koboldcpp colab.ipynb at concedo@f41f621e09822cc6ba48f3d791c41c239ebb6335 (git blob e47b699caed03db066575af01006d83067564b59); upstream license AGPL-3.0.

#@markdown ### Select a quick preset template (optional)
Template = "Fast smoke test - Qwen2.5-Coder 1.5B (Coding/API)" # @param ["Fast smoke test - Qwen2.5-Coder 1.5B (Coding/API)", "Balanced coder - Qwen2.5-Coder 7B (Coding/API)", "Stronger coder - Qwen2.5-Coder 14B (Coding/API)", "Large tight fit - Mistral Small 3.2 24B (Coding/API)"]


#@markdown ### Or, choose manually by <u>selecting checkboxes and dropdown</u> below (Overwrites above templates!)

#@markdown <hr>
LoadTextModel = False #@param {type:"boolean"}
Model = "https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF/resolve/f86cb2c1fa58255f8052cc32aeede1b7482d4361/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf" #@param ["https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF/resolve/f86cb2c1fa58255f8052cc32aeede1b7482d4361/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf", "https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct-GGUF/resolve/13fb94bfda8c8cf22497dc57b78f391a9acb426a/qwen2.5-coder-7b-instruct-q5_k_m.gguf", "https://huggingface.co/Qwen/Qwen2.5-Coder-14B-Instruct-GGUF/resolve/d0a692ef765eefbf2fabb130b3cb2e8917e3d225/qwen2.5-coder-14b-instruct-q4_k_m.gguf", "https://huggingface.co/unsloth/Mistral-Small-3.2-24B-Instruct-2506-GGUF/resolve/b750ec2299225e492f1bd27cab88a0a595fa848f/Mistral-Small-3.2-24B-Instruct-2506-Q4_K_S.gguf"] {"allow-input":true}
MdCommand = "" #@markdown <br>
Layers = "Auto" #@param ["Auto","999"]{allow-input: true}
ContextSize = "16384" #@param ["8192", "10240", "16384", "24576", "32768", "49152", "65536"] {allow-input: true}

#@markdown <hr>
LoadVisionMMProjector = False #@param {type:"boolean"}
Mmproj = "https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/resolve/main/mmproj-BF16.gguf" #@param ["https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/resolve/main/mmproj-BF16.gguf","https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/mmproj-BF16.gguf","https://huggingface.co/koboldcpp/mmproj/resolve/main/llama-13b-mmproj-v1.5.Q4_1.gguf","https://huggingface.co/koboldcpp/mmproj/resolve/main/mistral-7b-mmproj-v1.5-Q4_1.gguf","https://huggingface.co/koboldcpp/mmproj/resolve/main/llama-7b-mmproj-v1.5-Q4_0.gguf","https://huggingface.co/koboldcpp/mmproj/resolve/main/LLaMA3-8B_mmproj-Q4_1.gguf"]{allow-input: true}
VCommand = ""

#@markdown <hr>
LoadImgModel = False #@param {type:"boolean"}
ImgModel = "https://huggingface.co/koboldcpp/imgmodel/resolve/main/picx_real_q8_0.gguf" #@param ["https://huggingface.co/koboldcpp/imgmodel/resolve/main/picx_real_q8_0.gguf","https://huggingface.co/koboldcpp/imgmodel/resolve/main/dreamshaperXL_v2_q4_k.gguf"]{allow-input: true}
#@markdown <br>
ImgClip1 = "" #@param [""]{allow-input: true}
ImgVAE = "" #@param [""]{allow-input: true}
SCommand = ""

#@markdown <hr>
LoadSpeechModel = False #@param {type:"boolean"}
SpeechModel = "https://huggingface.co/koboldcpp/whisper/resolve/main/whisper-base.en-q5_1.bin" #@param ["https://huggingface.co/koboldcpp/whisper/resolve/main/whisper-base.en-q5_1.bin"]{allow-input: true}
WCommand = ""

#@markdown <hr>
LoadTTSModel = False #@param {type:"boolean"}
TTSModel = "https://huggingface.co/koboldcpp/tts/resolve/main/Kokoro_no_espeak_Q4.gguf" #@param ["https://huggingface.co/koboldcpp/tts/resolve/main/OuteTTS-0.2-500M-Q4_0.gguf","https://huggingface.co/koboldcpp/tts/resolve/main/Kokoro_no_espeak_Q4.gguf"]{allow-input: true}
WavTokModel = "" #@param ["https://huggingface.co/koboldcpp/tts/resolve/main/WavTokenizer-Large-75-Q4_0.gguf"]{allow-input: true}
TTSCommand = ""

#@markdown <hr>
LoadEmbeddingsModel = False #@param {type:"boolean"}
EmbeddingsModel = "https://huggingface.co/yixuan-chia/snowflake-arctic-embed-s-GGUF/resolve/main/snowflake-arctic-embed-s-Q4_0.gguf" #@param ["https://huggingface.co/yixuan-chia/snowflake-arctic-embed-s-GGUF/resolve/main/snowflake-arctic-embed-s-Q4_0.gguf"]{allow-input: true}
ECommand = ""

#@markdown <hr>
#@markdown Currently only AceStep 1.5 is supported, will use it if picked
LoadMusicModel = False #@param {type:"boolean"}
MusicModel = "--musiclowvram --musicllm https://huggingface.co/koboldcpp/music/resolve/main/acestep-5Hz-lm-1.7B-BF16.gguf --musicembeddings https://huggingface.co/koboldcpp/music/resolve/main/Ace-Qwen3-Embedding-0.6B-BF16.gguf --musicdiffusion https://huggingface.co/koboldcpp/music/resolve/main/acestep-v15-sftturbo50-Q8_0.gguf --musicvae https://huggingface.co/koboldcpp/music/resolve/main/ace-vae-BF16.gguf"
MuCommand = ""

#@markdown <hr>
#@markdown Deletes all models, forcing a fresh redownload when you restart colab.
DeleteExistingModels = False #@param {type:"boolean"}

#@markdown <hr>
#@markdown Enables multiplayer sessions, simply load the link and click join multiplayer.
Multiplayer = False #@param {type:"boolean"}
MPCommand = ""

#@markdown <hr>
#@markdown This enables saving stories directly to your google drive. You will have to grant permissions, and then you can access the saves from the "KoboldCpp Server Storage" option.
AllowSaveToGoogleDrive = False #@param {type:"boolean"}
SavGdriveCommand = ""

#@markdown <hr>
#@markdown Only select the following box if regular cloudflare tunnel fails to work. It will generate an inferior localtunnel tunnel, which you can use after entering a password.
MakeLocalTunnelFallback = False #@param {type:"boolean"}

#@markdown <hr>
#@markdown ### Dionysus coding/API settings
#@markdown These settings only add small fragments to the upstream launch line. Model downloading/loading below remains the official KoboldCpp cell logic.
KVCache = "q8_0" #@param ["f16", "bf16", "q8_0", "q5_1", "q4_0"]
FlashAttention = True #@param {type:"boolean"}
BatchSize = "512" #@param ["128", "256", "512", "1024", "2048", "4096"] {allow-input: true}
GenLimit = 0 #@param {type:"integer"}
ParallelRequests = "1" #@param ["1", "2", "4"] {allow-input: true}
UseMMQ = True #@param {type:"boolean"}
EnableWebSearch = False #@param {type:"boolean"}
UseRemoteTunnel = True #@param {type:"boolean"}
GenerateApiPassword = True #@param {type:"boolean"}
ExtraArgs = "" #@param {type:"string"}
KCommand = ""
FACommand = ""
BCommand = ""
GCommand = ""
PCommand = ""
CudaModeCommand = "--usecuda 0"
SearchCommand = ""
TunnelCommand = ""
ApiPassword = ""

CustomCtxSize = ""
CustomMmproj = ""
Customized = False
has_checked_box = any((LoadTextModel, LoadVisionMMProjector, LoadImgModel, LoadSpeechModel,LoadTTSModel, LoadEmbeddingsModel))
if not has_checked_box:
  Model = ""
  ImgModel = ""
  ImgClip1 = ""
  ImgVAE = ""
  Mmproj = ""
  if Template == "Fast smoke test - Qwen2.5-Coder 1.5B (Coding/API)":
    Customized = True
    Model = "https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF/resolve/f86cb2c1fa58255f8052cc32aeede1b7482d4361/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf"
    CustomCtxSize = "16384"
  if Template == "Balanced coder - Qwen2.5-Coder 7B (Coding/API)":
    Customized = True
    Model = "https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct-GGUF/resolve/13fb94bfda8c8cf22497dc57b78f391a9acb426a/qwen2.5-coder-7b-instruct-q5_k_m.gguf"
    CustomCtxSize = "32768"
  if Template == "Stronger coder - Qwen2.5-Coder 14B (Coding/API)":
    Customized = True
    Model = "https://huggingface.co/Qwen/Qwen2.5-Coder-14B-Instruct-GGUF/resolve/d0a692ef765eefbf2fabb130b3cb2e8917e3d225/qwen2.5-coder-14b-instruct-q4_k_m.gguf"
    CustomCtxSize = "32768"
  if Template == "Large tight fit - Mistral Small 3.2 24B (Coding/API)":
    Customized = True
    Model = "https://huggingface.co/unsloth/Mistral-Small-3.2-24B-Instruct-2506-GGUF/resolve/b750ec2299225e492f1bd27cab88a0a595fa848f/Mistral-Small-3.2-24B-Instruct-2506-Q4_K_S.gguf"
    CustomCtxSize = "10240"


if not Customized or Template=="" or has_checked_box:
  print("== Manual CHECKBOX OVERRIDE was detected, using manual selections ==")
  print(f"LoadTextModel: {LoadTextModel}")
  print(f"LoadVisionMMProjector: {LoadVisionMMProjector}")
  print(f"LoadImgModel: {LoadImgModel}")
  print(f"LoadSpeechModel: {LoadSpeechModel}")
  print(f"LoadTTSModel: {LoadTTSModel}")
  print(f"LoadEmbeddingsModel: {LoadEmbeddingsModel}")
elif Customized:
  print("== No checkbox override selected, loading a quick preset template! ==")
  print(f"CHOSEN PRSET: {Template}")
  print("If you want to load your own model selection, MAKE SURE you check one of the loader checkboxes...")
  if Model:
    LoadTextModel = True
    Layers = 999
  if ImgModel:
    LoadImgModel = True
  if CustomMmproj!="":
    Mmproj = CustomMmproj
    LoadVisionMMProjector = True
  if CustomCtxSize!="":
    ContextSize = CustomCtxSize

if Layers=="Auto":
  Layers = -1

print("=== STARTING THE LOAD ===")
if LoadTextModel:
  print(f"[Text Model] {Model}")
if LoadVisionMMProjector:
  print(f"[Vision MMProj] {Mmproj}")
if LoadImgModel:
  print(f"[Image Model] {ImgModel}")
  if ImgClip1:
    print(f"  └─ Clip: {ImgClip1}")
  if ImgVAE:
    print(f"  └─ VAE: {ImgVAE}")
if LoadSpeechModel:
  print(f"[Speech Model] {SpeechModel}")
if LoadTTSModel:
  print(f"[TTS Model] {TTSModel}")
  if WavTokModel:
    print(f"  └─ WavTokenizer: {WavTokModel}")
if LoadEmbeddingsModel:
  print(f"[Embeddings Model] {EmbeddingsModel}")

import os, glob
if not os.path.isfile("/opt/bin/nvidia-smi"):
  raise RuntimeError("⚠️Colab did not give you a GPU due to usage limits, this can take a few hours before they let you back in. Check out https://lite.koboldai.net for a free alternative (that does not provide an API link but can load KoboldAI saves and chat cards) or subscribe to Colab Pro for immediate access.⚠️")

if AllowSaveToGoogleDrive:
  print("Attempting to request access to save to your google drive...")
  try:
    from google.colab import drive
    import os, json
    drive.mount('/content/drive', force_remount=True)
    if not os.path.exists("/content/drive/MyDrive"):
      raise RuntimeError("Google Drive mount failed. Please grant permissions and try again.")
    kcppdir = '/content/drive/MyDrive/koboldcpp_data'
    os.makedirs(kcppdir, exist_ok=True)
    savedatapath = os.path.join(kcppdir, "koboldcpp_save_db.jsondb")
    if not os.path.exists(savedatapath):
        settings_data = {}
        with open(savedatapath, "w") as json_file:
            json.dump(settings_data, json_file, indent=4)
        print(f"Created new koboldcpp_save_db.jsondb at {savedatapath}")
    else:
        print(f"Loading saved data at {savedatapath}")
    SavGdriveCommand = f" --savedatafile {savedatapath}"
  except Exception as e:
    print(f"⚠️ Error: {e}")
    print("Please ensure you grant Google Drive permissions and try again.")

%cd /content
if Model and LoadTextModel:
  MdCommand = f"--model {Model}"
else:
  MdCommand = ""
if Mmproj and LoadVisionMMProjector:
  VCommand = f"--mmproj {Mmproj}"
else:
  VCommand = ""
if ImgModel and LoadImgModel:
  SCommand = f"--sdmodel {ImgModel} --sdthreads 4 --sdquant --sdclamped 768 --sdclipgpu"
  if ImgClip1 and ImgClip1!="":
    SCommand += f" --sdclip1 {ImgClip1}"
  if ImgVAE and ImgVAE!="":
    SCommand += f" --sdvae {ImgVAE}"
else:
  SCommand = ""
if SpeechModel and LoadSpeechModel:
  WCommand = f"--whispermodel {SpeechModel}"
else:
  WCommand = ""
if TTSModel and WavTokModel and LoadTTSModel:
  TTSCommand = f"--ttsmodel {TTSModel} --ttswavtokenizer {WavTokModel} --ttsgpu"
elif TTSModel and LoadTTSModel:
  TTSCommand = f"--ttsmodel {TTSModel} --ttsgpu"
else:
  TTSCommand = ""
if EmbeddingsModel and LoadEmbeddingsModel:
  ECommand = f"--embeddingsmodel {EmbeddingsModel}"
else:
  ECommand = ""
if LoadMusicModel:
  MuCommand = f"{MusicModel}"
else:
  MuCommand = ""
if Multiplayer:
  MPCommand = "--multiplayer"
else:
  MPCommand = ""

KCommand = f"--quantkv {KVCache}"
FACommand = "" if FlashAttention else "--noflashattention"
BCommand = f"--blasbatchsize {BatchSize}"
GCommand = f"--genlimit {GenLimit}" if int(GenLimit) > 0 else ""
PCommand = f"--parallelrequests {ParallelRequests}"
CudaModeCommand = "--usecuda 0" if UseMMQ else "--usecuda 0 nommq"
SearchCommand = "--websearch" if EnableWebSearch else ""
TunnelCommand = "--remotetunnel" if UseRemoteTunnel else ""
if GenerateApiPassword:
  import secrets
  ApiPassword = secrets.token_urlsafe(32)
  os.environ["KCPP_PASSWORD"] = ApiPassword
  os.environ["KCPP_ADMINPASSWORD"] = ApiPassword
  print("=" * 72)
  print(f"Dionysus API password / Bearer token for this run: {ApiPassword}")
  print("OpenAI-compatible base URL will be the printed tunnel URL + /v1. Copy this password, then clear cell outputs after startup.")
  print("=" * 72)
else:
  ApiPassword = ""
  os.environ.pop("KCPP_PASSWORD", None)
  os.environ.pop("KCPP_ADMINPASSWORD", None)
  print("WARNING: GenerateApiPassword is OFF; like the upstream default, no generated password will be set.")
print(f"[Dionysus settings] {CudaModeCommand} {KCommand} {FACommand} {BCommand} {GCommand} {PCommand} {SearchCommand} {TunnelCommand} ExtraArgs={ExtraArgs}")

if DeleteExistingModels:
  print("Deleting all cached models to redownload...")
  patterns = ['*.gguf', '*.bin', '*.ggml']
  for pattern in patterns:
      for file_path in glob.glob(pattern):
          try:
              os.remove(file_path)
              print(f"Deleted: {file_path}")
          except Exception as e:
              print(f"Failed to delete {file_path}: {e}")


!echo Downloading KoboldCpp, please wait...
!wget -O dlfile.tmp https://github.com/LostRuins/koboldcpp/releases/latest/download/koboldcpp-linux-x64 && mv dlfile.tmp koboldcpp_linux || (echo "Primary failed, trying mirror..." && wget -O dlfile.tmp https://koboldai.org/cpplinuxcu12 && mv dlfile.tmp koboldcpp_linux)
!test -f koboldcpp_linux && echo Download Successful || echo Download Failed
!chmod +x ./koboldcpp_linux
!apt update
!apt install aria2 -y

if MakeLocalTunnelFallback:
  import urllib
  print("Trying to use LocalTunnel as a fallback tunnel (not so good)...")
  ltpw = urllib.request.urlopen('https://loca.lt/mytunnelpassword').read().decode('utf8').strip("\n")
  !nohup npx --yes localtunnel --port 5001 > lt.log 2>&1 &
  !sleep 8
  print("=================")
  print("(LocalTunnel Results)")
  !cat lt.log
  print(f"Please open the above link, and input the password '{ltpw}'\nYour KoboldCpp will start shortly...")
  print("=================")
  !sleep 10
!./koboldcpp_linux $MdCommand $CudaModeCommand --gpulayers $Layers --contextsize $ContextSize $KCommand $FACommand $BCommand $GCommand $PCommand $SearchCommand --quiet $TunnelCommand $MPCommand $VCommand $SCommand $WCommand $TTSCommand $ECommand $MuCommand $SavGdriveCommand $ExtraArgs
